# Minggu 3 — Advanced ML (SVM RBF 5-Fold) & Deep Learning (MLP 128-64-32)

In [ ]:
import json
import time
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.svm import SVC
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(DATA_DIR / "train_model.csv")
val_df = pd.read_csv(DATA_DIR / "val_model.csv")

with open(DATA_DIR / "feature_lists.json", "r", encoding="utf-8") as f:
    feature_lists = json.load(f)

with open(DATA_DIR / "class_weights.json", "r", encoding="utf-8") as f:
    class_weights_info = json.load(f)

features_full = feature_lists["model_columns"]
hs_cols = ["hs_count", "hs_count_7d", "hs_count_30d", "hs_days_since_last"]
features_weather = [c for c in features_full if c not in hs_cols]
target_col = feature_lists["target"]

cw_dict = {0: class_weights_info["0"], 1: class_weights_info["1"]}
pos_weight_val = class_weights_info["scale_pos_weight"]

X_train = train_df[features_full].values
y_train = train_df[target_col].astype(int).values
X_val = val_df[features_full].values
y_val = val_df[target_col].astype(int).values

print(f"Data dimuat -> Train: {X_train.shape} | Validation: {X_val.shape}
")

def hitung_metrik(nama, y_true, y_pred, y_score, durasi):
    return {
        "Model": nama,
        "Durasi (dtk)": round(durasi, 2),
        "Accuracy": round(accuracy_score(y_true, y_pred), 4),
        "Precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_true, y_pred, zero_division=0), 4),
        "F1-Score": round(f1_score(y_true, y_pred, zero_division=0), 4),
        "PR-AUC": round(average_precision_score(y_true, y_score), 4),
        "ROC-AUC": round(roc_auc_score(y_true, y_score), 4),
    }

hasil_evaluasi = []

dt_baseline = joblib.load(MODELS_DIR / "baseline_decision_tree.pkl")
hasil_evaluasi.append(
    hitung_metrik(
        "1. Baseline: Decision Tree (Full)",
        y_val,
        dt_baseline.predict(val_df[features_full]),
        dt_baseline.predict_proba(val_df[features_full])[:, 1],
        0.0,
    )
)

print("1/2 Melatih SVM (RBF) 5-Fold Calibrated dengan class_weight...")
start_svm = time.time()
base_svc = SVC(kernel="rbf", class_weight=cw_dict, random_state=42)
svm_5fold = CalibratedClassifierCV(base_svc, cv=5, ensemble=False)
svm_5fold.fit(X_train, y_train)
durasi_svm = time.time() - start_svm
print(f"[SELESAI] SVM 5-Fold rampung dalam {durasi_svm:.2f} detik.
")

joblib.dump(svm_5fold, MODELS_DIR / "advanced_svm_rbf_5fold.pkl")
hasil_evaluasi.append(
    hitung_metrik(
        "2. Advanced: SVM RBF 5-Fold (Full)",
        y_val,
        svm_5fold.predict(X_val),
        svm_5fold.predict_proba(X_val)[:, 1],
        durasi_svm,
    )
)

print("2/2 Melatih Deep Learning MLP (128-64-32) dengan pos_weight...")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class TabularMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x)

def train_and_eval_mlp(X_tr, y_tr, X_va, epochs=40):
    torch.manual_seed(42)
    loader = DataLoader(
        TensorDataset(
            torch.tensor(X_tr, dtype=torch.float32),
            torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1),
        ),
        batch_size=64,
        shuffle=True,
    )
    model = TabularMLP(X_tr.shape[1]).to(device)
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pos_weight_val], dtype=torch.float32).to(device)
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    history_loss = []
    for _ in range(epochs):
        model.train()
        total_loss = 0.0
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        history_loss.append(total_loss / len(loader))

    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X_va, dtype=torch.float32).to(device))
        proba = torch.sigmoid(logits).cpu().numpy().ravel()
        pred = (proba >= 0.5).astype(int)
    return model, pred, proba, history_loss

start_mlp = time.time()
mlp_model, y_val_pred_mlp, y_val_proba_mlp, loss_hist = train_and_eval_mlp(
    X_train, y_train, X_val, epochs=40
)
durasi_mlp = time.time() - start_mlp
print(f"[SELESAI] MLP rampung dalam {durasi_mlp:.2f} detik (Loss akhir: {loss_hist[-1]:.4f}).
")

torch.save(mlp_model.state_dict(), MODELS_DIR / "advanced_mlp_full.pt")
hasil_evaluasi.append(
    hitung_metrik(
        "3. Deep Learning: MLP 128-64-32 (Full)",
        y_val,
        y_val_pred_mlp,
        y_val_proba_mlp,
        durasi_mlp,
    )
)

X_tr_w = train_df[features_weather].values
X_va_w = val_df[features_weather].values

t0 = time.time()
svm_w = CalibratedClassifierCV(
    SVC(kernel="rbf", class_weight=cw_dict, random_state=42), cv=5, ensemble=False
).fit(X_tr_w, y_train)
hasil_evaluasi.append(
    hitung_metrik(
        "Ablation: SVM 5-Fold (Cuaca Saja - 24 Fitur)",
        y_val,
        svm_w.predict(X_va_w),
        svm_w.predict_proba(X_va_w)[:, 1],
        time.time() - t0,
    )
)

t0 = time.time()
_, y_pred_mlp_w, y_proba_mlp_w, _ = train_and_eval_mlp(X_tr_w, y_train, X_va_w, epochs=40)
hasil_evaluasi.append(
    hitung_metrik(
        "Ablation: MLP 128-64-32 (Cuaca Saja - 24 Fitur)",
        y_val,
        y_pred_mlp_w,
        y_proba_mlp_w,
        time.time() - t0,
    )
)

df_hasil = pd.DataFrame(hasil_evaluasi)
print("=== PERBANDINGAN SKOR VALIDASI (FULL 28 FITUR VS ABLATION 24 FITUR) ===")
print(df_hasil.to_string(index=False))

print("
--- Classification Report SVM 5-Fold (Full) ---")
print(classification_report(y_val, svm_5fold.predict(X_val)))

print("--- Classification Report MLP 128-64-32 (Full) ---")
print(classification_report(y_val, y_val_pred_mlp))
